In [ ]:
GPU="0"
num_GPUs = 1
gen_image_batch_size=16

In [ ]:
import os
os.environ['PATH'] += ':/home/temp0/anaconda3/envs/edm2/bin'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

def show_images_from_dir(folder_path, num_images=4, start_seed=42):
    """
    顯示指定資料夾中的圖片（根據檔名排序，從指定種子碼開始），以 2x2 格顯示。

    Args:
        folder_path (str): 圖片所在資料夾路徑
        num_images (int): 要顯示的圖片數量（預設 4）
        start_seed (int): 從第幾個種子（圖片）開始（根據檔名排序）
    """
    if not os.path.isdir(folder_path):
        print(f"[錯誤] 資料夾不存在：{folder_path}")
        return

    # 過濾圖片檔案，並根據檔名中的數字排序
    image_files = sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
    ], key=lambda x: int(''.join(filter(str.isdigit, os.path.basename(x))) or 0))

    # 找到起始 index
    start_index = next((i for i, f in enumerate(image_files)
                        if int(''.join(filter(str.isdigit, os.path.basename(f))) or 0) >= start_seed), None)

    if start_index is None:
        print(f"[警告] 找不到 seed >= {start_seed} 的圖片。")
        return

    subset = image_files[start_index:start_index + num_images]
    if not subset:
        print(f"[警告] 從 seed {start_seed} 起沒有足夠圖片可顯示。")
        return

    # 顯示 2x2 圖片
    rows = cols = 2
    plt.figure(figsize=(8, 8))
    for i, img_path in enumerate(subset):
        img = Image.open(img_path)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
num_images = 1
w=1.15
num_steps = 250
dirname=f'/data/guidance-team/out/2025_0911_DiT'
!CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl: torchrun \
--standalone --nproc_per_node={num_GPUs} ./edm2impl/prob_checking_DiT.py \
--outdir=/{dirname} \
--subdirs \
--seeds=0-{num_images} \
--sampler=cfg \
--guidance_scheduler=const_scheduler \
--classifier_path=/data/guidance-team-new/Imagenet64_res50_9/checkpoint_epoch17_batch180000.pt \
--save_classifier_stats=True \
--debug=True \
--heun_guid=True \
--guidance={w} \
--steps={num_steps} \
--batch={gen_image_batch_size}
show_images_from_dir(f"/{dirname}/000000", start_seed=42)
! CUDA_VISIBLE_DEVICES={GPU} PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 -- \
./base/calculate_metrics.py calc \
--images={dirname} \
--ref=https://nvlabs-fi-cdn.nvidia.com/edm2/dataset-refs/img64.pkl \
--num={num_images} \
--metrics='fid' \
--batch=4
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_is.py \
--data_root /{dirname} --cuda --resize \
--batch_size 32 \
--splits 10 --max_images {num_images}
! PYTHONWARNINGS="ignore" CUDA_VISIBLE_DEVICES={GPU} \
PYTHONPATH=.:./base:./classifer:./edm2impl torchrun \
--standalone --nproc_per_node=1 ./edm2impl/calculate_ms_ssim.py \
--data_root {dirname} --cuda --size 256 \
--batch_size=128 \
--num_pairs=5000

In [ ]:
def space_timesteps(num_timesteps, section_counts):
    """
    Create a list of timesteps to use from an original diffusion process,
    given the number of timesteps we want to take from equally-sized portions
    of the original process.
    For example, if there's 300 timesteps and the section counts are [10,15,20]
    then the first 100 timesteps are strided to be 10 timesteps, the second 100
    are strided to be 15 timesteps, and the final 100 are strided to be 20.
    If the stride is a string starting with "ddim", then the fixed striding
    from the DDIM paper is used, and only one section is allowed.
    :param num_timesteps: the number of diffusion steps in the original
                          process to divide up.
    :param section_counts: either a list of numbers, or a string containing
                           comma-separated numbers, indicating the step count
                           per section. As a special case, use "ddimN" where N
                           is a number of steps to use the striding from the
                           DDIM paper.
    :return: a set of diffusion steps from the original process to use.
    """
    if isinstance(section_counts, str):
        if section_counts.startswith("ddim"):
            desired_count = int(section_counts[len("ddim") :])
            for i in range(1, num_timesteps):
                if len(range(0, num_timesteps, i)) == desired_count:
                    return set(range(0, num_timesteps, i))
            raise ValueError(
                f"cannot create exactly {num_timesteps} steps with an integer stride"
            )
        section_counts = [int(x) for x in section_counts.split(",")]
    size_per = num_timesteps // len(section_counts)
    extra = num_timesteps % len(section_counts)
    start_idx = 0
    all_steps = []
    for i, section_count in enumerate(section_counts):
        size = size_per + (1 if i < extra else 0)
        if size < section_count:
            raise ValueError(
                f"cannot divide section of {size} steps into {section_count}"
            )
        if section_count <= 1:
            frac_stride = 1
        else:
            frac_stride = (size - 1) / (section_count - 1)
        cur_idx = 0.0
        taken_steps = []
        for _ in range(section_count):
            taken_steps.append(start_idx + round(cur_idx))
            cur_idx += frac_stride
        all_steps += taken_steps
        start_idx += size
    return set(all_steps)
len(space_timesteps(1000, "250"))

In [ ]:
import math
import numpy as np
def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    """
    Create a beta schedule that discretizes the given alpha_t_bar function,
    which defines the cumulative product of (1-beta) over time from t = [0,1].
    :param num_diffusion_timesteps: the number of betas to produce.
    :param alpha_bar: a lambda that takes an argument t from 0 to 1 and
                      produces the cumulative product of (1-beta) up to that
                      part of the diffusion process.
    :param max_beta: the maximum beta to use; use values lower than 1 to
                     prevent singularities.
    """
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)
alpha_bar = lambda t: math.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
alpha_bar2 = lambda j: math.sin(j/M/(1+C2)* math.pi / 2)**2

In [ ]:
alpha_bar2(1000)

In [ ]:
alpha_bar2(0)

In [ ]:
alpha_bar(0)

In [ ]:
M = 1000
C1 = 0.001
C2 = 0.008
# alpha_bar2 = lambda t: math.sin((1-t)/1.008* math.pi / 2)**2
alpha_bar2 = lambda j: math.sin(j/(M*(C2+1))* math.pi / 2)**2
# [(np.sin(np.pi/2*j/(M*(C2+1)))**2,math.cos(((M-j-1)/M + C2) / (1+C2) * math.pi / 2) ** 2) for j in (range(1000))][::-20]
u = [0]
for j in range(1000,1,-1):
    # print(j)
    A = alpha_bar2(j-1) / alpha_bar2(j)
    # print(A)
    B = max(A,C1)
    # print(B)
    C = (u[-1]**2+1)/(B)
    # print(C)
    D = np.sqrt(C - 1)
    # print(D)
    u.append(D)
u[::10]
# print(u)
import matplotlib.pyplot as plt

plt.plot(u)
plt.show()

In [ ]:
sum = 0
for v in u:
    if v >0.34 and v <=1.02:
        sum+=1
sum

In [ ]:
M, N, C1, C2 = 1000, 250, 0.001, 0.008
u_array = [0 for i in range(M+1)]

def alpha_bar(j):
    return np.sin(np.pi / 2 * j / (M * (C2 + 1))) ** 2

for j in range(M-1, -1, -1):
    alpha_bar_j = alpha_bar(j)
    alpha_bar_j_plus_1 = alpha_bar(j + 1)
    u_array[j] = np.sqrt((u_array[j + 1] ** 2 + 1) / max(alpha_bar_j / alpha_bar_j_plus_1, C1) - 1)

actual_sigma = []

In [ ]:
u_array